In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import csr_matrix

# Step 1 ( Load Data ) :

In [2]:

df = pd.read_csv('smart_cleaned.csv')
df['DateTime'] = pd.to_datetime(df['DateTime'])
print("Loaded:", df.shape)

Loaded: (1730592, 6)


# Step 2 ( Encode ) :

In [3]:
user_encoder     = LabelEncoder()
product_encoder  = LabelEncoder()
category_encoder = LabelEncoder()

In [4]:
df['user_idx']     = user_encoder.fit_transform(df['User_ID'])
df['product_idx']  = product_encoder.fit_transform(df['Product_ID'])
df['category_idx'] = category_encoder.fit_transform(df['Category_ID'])

print(f"\nUnique Users     : {df['user_idx'].nunique():,}")
print(f"Unique Products  : {df['product_idx'].nunique():,}")
print(f"Unique Categories: {df['category_idx'].nunique():,}")


Unique Users     : 19,995
Unique Products  : 590,266
Unique Categories: 6,484


# Step 3 ( User-Item Matrix ) :

In [5]:
n_users    = df['user_idx'].nunique()
n_products = df['product_idx'].nunique()

user_item_matrix = csr_matrix(
    (df['Score'], (df['user_idx'], df['product_idx'])),
    shape=(n_users, n_products)
)

print(f"\nUser-Item Matrix : {user_item_matrix.shape}")
print(f"Matrix Density   : {user_item_matrix.nnz / (n_users * n_products) * 100:.4f}%")


User-Item Matrix : (19995, 590266)
Matrix Density   : 0.0114%


# Step 4 ( Hybrid Model ) :


In [6]:
print("Training model...")

svd = TruncatedSVD(
    n_components=300,
    random_state=42,
    n_iter=15
)

user_factors = svd.fit_transform(user_item_matrix)
item_factors = svd.components_.T

print(f"✅ Model trained!")
print(f"User Factors Shape : {user_factors.shape}")
print(f"Item Factors Shape : {item_factors.shape}")
print(f"Explained Variance : {svd.explained_variance_ratio_.sum()*100:.2f}%")






Training model...
✅ Model trained!
User Factors Shape : (19995, 300)
Item Factors Shape : (590266, 300)
Explained Variance : 11.95%


# Step 5 ( User Category Profile )

In [7]:
user_category_profile = df.groupby(
    ['User_ID', 'Category_ID'])['Score'].sum().reset_index()
user_category_profile.columns = ['User_ID', 'Category_ID', 'Category_Score']
print(f"\nUser Category Profile: {user_category_profile.shape}")


User Category Profile: (441971, 3)


# Step 6 ( Hybrid Recommendation Function ) 


In [8]:
def get_hybrid_recommendations(user_id, n=5, svd_weight=0.7, cat_weight=0.3):

    user_idx    = user_encoder.transform([user_id])[0]
    user_vector = user_factors[user_idx]
    user_norm   = user_vector / (np.linalg.norm(user_vector) + 1e-10)
    item_norms  = item_factors / (np.linalg.norm(item_factors, axis=1, keepdims=True) + 1e-10)
    svd_scores  = user_norm @ item_norms.T

    # Category Score
    user_cats = user_category_profile[
        user_category_profile['User_ID'] == user_id
    ].sort_values('Category_Score', ascending=False)['Category_ID'].values

    cat_scores = np.zeros(n_products)
    for cat in user_cats[:3]:
        cat_products = df[df['Category_ID'] == cat]['product_idx'].values
        cat_scores[cat_products] += 1

    if cat_scores.max() > 0:
        cat_scores = cat_scores / cat_scores.max()

    # Hybrid Score
    hybrid_scores = (svd_weight * svd_scores) + (cat_weight * cat_scores)

    # استبعاد المنتجات اللي شافها
    seen = df[df['User_ID'] == user_id]['product_idx'].values
    hybrid_scores[seen] = -1

    top_indices  = hybrid_scores.argsort()[::-1][:n]
    top_products = product_encoder.inverse_transform(top_indices)
    top_cats     = [
        df[df['product_idx'] == idx]['Category_ID'].iloc[0]
        if len(df[df['product_idx'] == idx]) > 0 else None
        for idx in top_indices
    ]

    return pd.DataFrame({
        'Product_ID'  : top_products,
        'Category_ID' : top_cats,
        'Hybrid_Score': hybrid_scores[top_indices].round(4)
    })


# Step 7 ( Test )


In [9]:
sample_user = df['User_ID'].iloc[0]
print(f"\n=== Recommendations for User {sample_user} ===")
print(get_hybrid_recommendations(sample_user))

print("\n=== Testing on 3 different users ===")
for user in df['User_ID'].sample(3, random_state=42).values:
    print(f"\nUser {user}:")
    print(get_hybrid_recommendations(user))


=== Recommendations for User 1000283 ===
   Product_ID  Category_ID  Hybrid_Score
0     4759222      2733371        0.9285
1     2002818      2733371        0.9123
2      322300      2733371        0.9123
3     1398257      2733371        0.9105
4      559411      2733371        0.8552

=== Testing on 3 different users ===

User 589014:
   Product_ID  Category_ID  Hybrid_Score
0     3876767      3566895        0.8808
1      349886      2154669        0.6270
2      170755      1029459        0.6228
3     1081117      2885642        0.6228
4     1986307      1879194        0.6228

User 969882:
   Product_ID  Category_ID  Hybrid_Score
0      905208      1851156        0.7550
1     1182055      1851156        0.7452
2       27514      1851156        0.6380
3     1183349      1851156        0.6380
4     4890143      1851156        0.6184

User 209831:
   Product_ID  Category_ID  Hybrid_Score
0     3131664       360294        0.6054
1     1097463       360294        0.5816
2     3516050    

# Evaluation :

### Step 1 ( Split Data Train / Test ) :

In [10]:
from sklearn.model_selection import train_test_split

buy_df = df[df['Behavior'] == 'buy'].copy()

train_buy, test_buy = train_test_split(
    buy_df,
    test_size=0.2,
    random_state=42
)

print(f"Users with buy : {buy_df['User_ID'].nunique():,}")
print(f"Train buy      : {len(train_buy):,}")
print(f"Test buy       : {len(test_buy):,}")

Users with buy : 13,005
Train buy      : 29,360
Test buy       : 7,340


### Step 2 ( Category-Based Evaluation Function )


In [11]:
def evaluate_by_category(test_data, k=5):
    precisions = []
    recalls    = []

    test_users = test_data['User_ID'].unique()

    for user_id in test_users:
        actual_cats = set(
            test_data[test_data['User_ID'] == user_id]['Category_ID'].values
        )

        try:
            recs = get_hybrid_recommendations(user_id, n=k)
            recommended_cats = set(recs['Category_ID'].values)
        except:
            continue

        precision = len(actual_cats & recommended_cats) / k
        recall    = len(actual_cats & recommended_cats) / len(actual_cats) if actual_cats else 0

        precisions.append(precision)
        recalls.append(recall)

    avg_p = np.mean(precisions)
    avg_r = np.mean(recalls)

    return {
        'Precision@K'    : avg_p,
        'Recall@K'       : avg_r,
        'F1@K'           : 2 * avg_p * avg_r / (avg_p + avg_r + 1e-10),
        'Users evaluated': len(precisions)
    }



### Step 3 ( Evaluate with Different K values )

In [12]:
# ============================================
# Step 3: تقييم بـ K مختلفة
# ============================================
print("\n=== Category-Based Evaluation ===")
for k in [5, 10, 20]:
    results = evaluate_by_category(test_buy, k=k)
    print(f"\nK = {k}:")
    print(f"  Precision@{k} : {results['Precision@K']*100:.2f}%")
    print(f"  Recall@{k}    : {results['Recall@K']*100:.2f}%")
    print(f"  F1@{k}        : {results['F1@K']:.4f}")
    print(f"  Users evaluated: {results['Users evaluated']:,}")


=== Category-Based Evaluation ===

K = 5:
  Precision@5 : 8.44%
  Recall@5    : 34.13%
  F1@5        : 0.1353
  Users evaluated: 5,285

K = 10:
  Precision@10 : 4.92%
  Recall@10    : 39.65%
  F1@10        : 0.0876
  Users evaluated: 5,285

K = 20:
  Precision@20 : 2.83%
  Recall@20    : 45.32%
  F1@20        : 0.0532
  Users evaluated: 5,285
